# Domain D: Spatial Organization and Micro-Architecture

This notebook addresses two questions:
- **D1:** Is there spatial clustering of functionally similar neurons beyond what cell-type identity predicts?
- **D2:** Does cell-type composition vary across the columnar extent and relate to functional properties?

**Data:** Zarr multimodal stores with 3D cell coordinates (x_loc, y_loc, z_loc), cell-type labels, ΔF/F traces, and gene expression.

**Note:** Additional high-precision 3D centroids, soma volume (n_voxels), soma elongation (size_pc1/2/3_um), and orientation angle are available in the zarr multimodal stores under `morphology/mask_properties/`. See Domain H notebook for morphology-specific analyses.

In [ ]:
import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from types import SimpleNamespace
from scipy.stats import kruskal, spearmanr, pearsonr, chi2_contingency
from scipy.spatial.distance import cdist, pdist, squareform
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings('ignore')

sns.set_context('talk')
sns.set_style('whitegrid')

# ══════════════════════════════════════════════════════════════════════
# Load data from zarr multimodal stores
# ══════════════════════════════════════════════════════════════════════
ZARR_DIR = 'multimodal_data'
MOUSE_IDS = ['778174', '786297', '797371']
SESSIONS = ['session_1', 'session_2', 'session_3']

def load_mouse_zarr(mouse_id):
    """Load one mouse's data from zarr, returning an adata-like SimpleNamespace."""
    z = zarr.open(f'{ZARR_DIR}/{mouse_id}_multimodal_data.zarr', 'r')
    ct = z['transcriptomics/cell_type']
    morph = z['morphology/mask_properties']
    obs_dict = {
        'unique_id': z['unique_id'][:].astype(str),
        'mouse_id': mouse_id,
        'class_name': ct['class_name'][:],
        'subclass_name': ct['subclass_name'][:],
        'subclass_label': ct['subclass_label'][:],
        'supertype_name': ct['supertype_name'][:],
        'cluster_name': ct['cluster_name'][:],
        'cluster_alias': ct['cluster_alias'][:],
        'x_loc': morph['centroid_x_um'][:],
        'y_loc': morph['centroid_y_um'][:],
        'z_loc': morph['centroid_z_um'][:],
    }
    obs_df = pd.DataFrame(obs_dict)
    n_cells = len(obs_df)

    X_sessions, var_sessions = [], []
    for si, sess in enumerate(SESSIONS):
        gs = z[f'ophys/drifting_gratings/{sess}/stim_aligned_dff/GratingStim']
        dff = gs['dff'][:]
        time_rel = gs['time_relative'][:]
        running = gs['running'][:]
        gray = gs['trial_info/gray_screen'][:]
        valid = ~gray
        stim_mask = (time_rel >= 0) & (time_rel <= 2.0)
        dff_avg = dff[valid][:, stim_mask, :].mean(axis=1)
        run_speed = running[valid][:, stim_mask, 0].mean(axis=1)
        X_sessions.append(dff_avg.T)
        var_sessions.append(pd.DataFrame({
            'contrast': gs['trial_info/contrast'][:][valid],
            'orientation': gs['trial_info/orientation'][:][valid],
            'temporal_frequency': gs['trial_info/temporal_frequency'][:][valid],
            'spatial_frequency': gs['trial_info/spatial_frequency'][:][valid],
            'stim_block': gs['trial_info/stim_block'][:][valid],
            'stim_name': gs['trial_info/stim_name'][:][valid],
            'start_time': gs['trial_info/start_time'][:][valid],
            'stop_time': gs['trial_info/stop_time'][:][valid],
            'duration': gs['trial_info/duration'][:][valid],
            'avg_running': run_speed,
            'is_running': run_speed > 1.0,
            'day': si + 1,
        }))
    return SimpleNamespace(
        X=np.hstack(X_sessions), obs=obs_df, var=pd.concat(var_sessions, ignore_index=True),
        n_obs=n_cells, n_vars=sum(v.shape[0] for v in var_sessions)
    )

adata_list = [load_mouse_zarr(mid) for mid in MOUSE_IDS]

obs = pd.concat([a.obs for a in adata_list], ignore_index=True)
X = np.vstack([a.X for a in adata_list])
var = adata_list[0].var.copy()

SUBCLASS_ORDER = ['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut', '022 L5 ET CTX Glut',
                  '052 Pvalb Gaba', '053 Sst Gaba', '046 Vip Gaba', '049 Lamp5 Gaba']
SUBCLASS_COLORS = {'007 L2/3 IT CTX Glut': '#1f77b4', '006 L4/5 IT CTX Glut': '#2ca02c',
                   '022 L5 ET CTX Glut': '#9467bd', '052 Pvalb Gaba': '#d62728',
                   '053 Sst Gaba': '#ff7f0e', '046 Vip Gaba': '#e377c2', '049 Lamp5 Gaba': '#8c564b'}
SUBCLASS_SHORT = {'007 L2/3 IT CTX Glut': 'L2/3 IT', '006 L4/5 IT CTX Glut': 'L4/5 IT',
                  '022 L5 ET CTX Glut': 'L5 ET', '052 Pvalb Gaba': 'Pvalb',
                  '053 Sst Gaba': 'Sst', '046 Vip Gaba': 'Vip', '049 Lamp5 Gaba': 'Lamp5'}
present_subclasses = [s for s in SUBCLASS_ORDER if s in obs['subclass_name'].unique()]

# Backward-compatible adata object for downstream cells
adata = SimpleNamespace(X=X, obs=obs, var=var, n_obs=len(obs), n_vars=X.shape[1])

orientations = np.array([0, 45, 90, 135, 180, 225, 270, 315])

# ── 3D coordinates ──
coords = obs[['x_loc', 'y_loc', 'z_loc']].values.astype(float)
print(f"Total cells: {len(obs)}")
print(f"Coordinate ranges: x=[{coords[:,0].min():.0f}, {coords[:,0].max():.0f}], "
      f"y=[{coords[:,1].min():.0f}, {coords[:,1].max():.0f}], "
      f"z=[{coords[:,2].min():.0f}, {coords[:,2].max():.0f}]")

## D1: Spatial Clustering of Functionally Similar Neurons

Compute Moran's I for orientation preference, signal/noise correlations vs pairwise distance, and test whether functional similarity at short range exceeds what cell-type identity alone predicts.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# D1.1  Compute tuning vectors for orientation preference
# ══════════════════════════════════════════════════════════════════════

# Orientation tuning from contrast-context blocks
contrast_blocks = var['stim_block'].isin([0.0, 2.0])
ori_responses = np.zeros((adata.n_obs, len(orientations)))
for i, ori in enumerate(orientations):
    mask = contrast_blocks & (var['orientation'] == ori)
    trial_idx = np.where(mask.values)[0]
    if len(trial_idx) > 0:
        ori_responses[:, i] = np.nanmean(X[:, trial_idx], axis=1)

# Preferred orientation
def preferred_orientation(responses, orientations_deg):
    theta = np.deg2rad(2 * orientations_deg)
    vec = np.sum(responses * np.exp(1j * theta))
    return np.rad2deg(np.angle(vec)) / 2 % 180

def compute_osi(responses, orientations_deg):
    theta = np.deg2rad(2 * orientations_deg)
    return np.abs(np.sum(responses * np.exp(1j * theta))) / np.sum(np.abs(responses))

pref_ori = np.array([preferred_orientation(ori_responses[i], orientations) for i in range(adata.n_obs)])
osi = np.array([compute_osi(ori_responses[i], orientations) for i in range(adata.n_obs)])

print(f"Preferred orientation range: {np.nanmin(pref_ori):.1f}° to {np.nanmax(pref_ori):.1f}°")
print(f"OSI range: {np.nanmin(osi):.3f} to {np.nanmax(osi):.3f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# D1.2  Moran's I: Spatial autocorrelation of orientation preference
# ══════════════════════════════════════════════════════════════════════

def morans_i(values, coords, distance_threshold):
    """Compute Moran's I for values given coordinates and a distance threshold."""
    n = len(values)
    mean_v = np.nanmean(values)
    dev = values - mean_v
    
    # Build binary weight matrix (1 if within threshold)
    dist_mat = squareform(pdist(coords))
    W = (dist_mat < distance_threshold) & (dist_mat > 0)
    W_sum = W.sum()
    
    if W_sum == 0:
        return np.nan
    
    numerator = 0
    for i in range(n):
        for j in range(n):
            if W[i, j]:
                numerator += dev[i] * dev[j]
    
    denominator = np.sum(dev**2)
    I = (n / W_sum) * (numerator / denominator) if denominator > 0 else 0
    return I

# Compute Moran's I at multiple distance thresholds (per mouse to avoid cross-mouse artifacts)
distance_bins = [30, 50, 75, 100, 150, 200, 300]
mouse_ids = obs['mouse_id'].unique()

morans_results = []
for mouse in mouse_ids:
    mouse_mask = obs['mouse_id'].values == mouse
    mouse_coords = coords[mouse_mask]
    mouse_pref = pref_ori[mouse_mask]
    mouse_osi = osi[mouse_mask]
    
    # Use circular variable for orientation: cos(2*pref_ori) and sin(2*pref_ori)
    cos_pref = np.cos(np.deg2rad(2 * mouse_pref))
    sin_pref = np.sin(np.deg2rad(2 * mouse_pref))
    
    for d in distance_bins:
        I_cos = morans_i(cos_pref, mouse_coords, d)
        I_osi = morans_i(mouse_osi, mouse_coords, d)
        morans_results.append({'mouse': mouse, 'distance': d, 'measure': 'cos(2θ)', 'I': I_cos})
        morans_results.append({'mouse': mouse, 'distance': d, 'measure': 'OSI', 'I': I_osi})

morans_df = pd.DataFrame(morans_results)

# ── Visualization: Moran's I correlogram ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, measure in zip(axes, ['cos(2θ)', 'OSI']):
    sub = morans_df[morans_df['measure'] == measure]
    for mouse in mouse_ids:
        msub = sub[sub['mouse'] == mouse]
        ax.plot(msub['distance'], msub['I'], 'o-', alpha=0.5, label=f'Mouse {mouse}')
    # Mean across mice
    mean_I = sub.groupby('distance')['I'].mean()
    ax.plot(mean_I.index, mean_I.values, 'k-', linewidth=3, label='Mean')
    ax.axhline(0, color='gray', ls='--', alpha=0.5)
    ax.set_xlabel('Distance threshold (µm)')
    ax.set_ylabel("Moran's I")
    ax.set_title(f"D1: Moran's I — {measure}", fontweight='bold')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# ── Permutation test for significance ──
n_perm = 200
print("\nMoran's I permutation test (d=50µm):")
for mouse in mouse_ids:
    mouse_mask = obs['mouse_id'].values == mouse
    mouse_coords = coords[mouse_mask]
    cos_pref = np.cos(np.deg2rad(2 * pref_ori[mouse_mask]))
    
    observed_I = morans_i(cos_pref, mouse_coords, 50)
    null_I = []
    for _ in range(n_perm):
        shuffled = np.random.permutation(cos_pref)
        null_I.append(morans_i(shuffled, mouse_coords, 50))
    
    p_val = np.mean(np.array(null_I) >= observed_I)
    print(f"  Mouse {mouse}: I={observed_I:.4f}, p={p_val:.4f} (permutation)")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# D1.3  Signal & Noise correlation vs pairwise distance
# ══════════════════════════════════════════════════════════════════════

# Use a single mouse to keep computation tractable
mouse_pick = mouse_ids[np.argmax([np.sum(obs['mouse_id'].values == m) for m in mouse_ids])]
m_mask = obs['mouse_id'].values == mouse_pick
m_X = X[m_mask]
m_coords = coords[m_mask]
m_ori_resp = ori_responses[m_mask]
m_subclass = obs['subclass_name'].values[m_mask]
n_m = m_mask.sum()
print(f"Computing correlations for mouse {mouse_pick} ({n_m} cells)")

# Signal correlation: correlation of trial-averaged tuning vectors
signal_corr = np.corrcoef(m_ori_resp)

# Noise correlation: trial-by-trial residuals
# For each condition (orientation), subtract the trial-averaged response, then correlate residuals
contrast_blocks_m = var['stim_block'].isin([0.0, 2.0])
residuals_all = []
for ori in orientations:
    mask = contrast_blocks_m & (var['orientation'] == ori)
    trial_idx = np.where(mask.values)[0]
    if len(trial_idx) > 5:
        trial_resp = m_X[:, trial_idx]  # cells x trials
        mean_resp = np.nanmean(trial_resp, axis=1, keepdims=True)
        residuals_all.append(trial_resp - mean_resp)

if residuals_all:
    residuals_cat = np.hstack(residuals_all)  # cells x total_trials
    noise_corr = np.corrcoef(residuals_cat)
else:
    noise_corr = np.full((n_m, n_m), np.nan)

# Pairwise distance
dist_mat = squareform(pdist(m_coords))

# ── Bin by distance and plot ──
distance_edges = np.arange(0, 400, 25)
distance_centers = (distance_edges[:-1] + distance_edges[1:]) / 2

# Get upper triangle indices
triu_i, triu_j = np.triu_indices(n_m, k=1)
pair_dist = dist_mat[triu_i, triu_j]
pair_signal = signal_corr[triu_i, triu_j]
pair_noise = noise_corr[triu_i, triu_j]
pair_same_type = m_subclass[triu_i] == m_subclass[triu_j]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, corr_vals, title in zip(axes[:2], [pair_signal, pair_noise],
                                  ['Signal Correlation', 'Noise Correlation']):
    mean_all, mean_same, mean_diff = [], [], []
    for b in range(len(distance_edges)-1):
        bin_mask = (pair_dist >= distance_edges[b]) & (pair_dist < distance_edges[b+1])
        valid = bin_mask & ~np.isnan(corr_vals)
        if valid.sum() > 10:
            mean_all.append(np.nanmean(corr_vals[valid]))
            mean_same.append(np.nanmean(corr_vals[valid & pair_same_type]))
            mean_diff.append(np.nanmean(corr_vals[valid & ~pair_same_type]))
        else:
            mean_all.append(np.nan)
            mean_same.append(np.nan)
            mean_diff.append(np.nan)
    
    ax.plot(distance_centers, mean_all, 'k-', linewidth=2, label='All pairs')
    ax.plot(distance_centers, mean_same, 'r-', linewidth=2, label='Same subclass')
    ax.plot(distance_centers, mean_diff, 'b-', linewidth=2, label='Different subclass')
    ax.set_xlabel('Pairwise Distance (µm)')
    ax.set_ylabel(title)
    ax.set_title(f'D1: {title} vs Distance', fontweight='bold')
    ax.legend(fontsize=8)
    ax.axhline(0, color='gray', ls='--', alpha=0.4)

# Residual signal correlation after regressing out cell-type identity
# (test if functional similarity beyond type depends on distance)
ax = axes[2]
# Within same-type pairs only
same_signal = pair_signal[pair_same_type & ~np.isnan(pair_signal)]
same_dist = pair_dist[pair_same_type & ~np.isnan(pair_signal)]
r, p = spearmanr(same_dist, same_signal)
ax.scatter(same_dist, same_signal, alpha=0.05, s=5, c='gray')
# Binned mean
for b in range(len(distance_edges)-1):
    bin_mask = (same_dist >= distance_edges[b]) & (same_dist < distance_edges[b+1])
    if bin_mask.sum() > 5:
        ax.scatter(distance_centers[b], np.mean(same_signal[bin_mask]), c='red', s=40, zorder=5)
ax.set_xlabel('Distance (µm)')
ax.set_ylabel('Signal Correlation')
ax.set_title(f'D1: Within-Type Signal Corr vs Dist (ρ={r:.3f}, p={p:.1e})', fontweight='bold')
ax.axhline(0, color='gray', ls='--', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# D1.4  Spatial maps of orientation preference (per depth)
# ══════════════════════════════════════════════════════════════════════

# Color by preferred orientation using a circular colormap
from matplotlib.colors import hsv_to_rgb

def ori_to_color(ori_deg):
    """Map orientation (0-180°) to hue."""
    hue = (ori_deg % 180) / 180
    return hsv_to_rgb([hue, 0.8, 0.9])

# Get unique planes per mouse
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flat

# Only use cells with reasonable OSI
well_tuned = osi > 0.1

for mi, mouse in enumerate(mouse_ids):
    ax = axes[mi]
    m_mask_t = (obs['mouse_id'].values == mouse) & well_tuned
    x = coords[m_mask_t, 0]
    y = coords[m_mask_t, 1]
    z = coords[m_mask_t, 2]
    po = pref_ori[m_mask_t]
    
    colors = np.array([ori_to_color(o) for o in po])
    sc = ax.scatter(x, y, c=colors, s=15, alpha=0.7)
    ax.set_xlabel('X (µm)')
    ax.set_ylabel('Y (µm)')
    ax.set_title(f'Mouse {mouse} (n={m_mask_t.sum()})', fontweight='bold')
    ax.set_aspect('equal')

# Colorbar legend
ax_cb = axes[-1]
ax_cb.axis('off')
gradient = np.linspace(0, 180, 100)
colors_bar = np.array([ori_to_color(o) for o in gradient])
for i, (o, c) in enumerate(zip(gradient, colors_bar)):
    ax_cb.barh(o, 1, color=c, height=2)
ax_cb.set_ylabel('Preferred Orientation (°)')
ax_cb.set_title('Color Legend')
ax_cb.set_xlim(0, 1.5)
ax_cb.yaxis.set_visible(True)

plt.suptitle('D1: Spatial Maps of Preferred Orientation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Layer-specific analysis: L2/3 vs L4/5 ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (layer, sc_name) in zip(axes, [('L2/3', '007 L2/3 IT CTX Glut'),
                                         ('L4/5', '006 L4/5 IT CTX Glut')]):
    layer_mask = (obs['subclass_name'].values == sc_name) & well_tuned
    if layer_mask.sum() < 10:
        ax.set_visible(False)
        continue
    layer_dist = pdist(coords[layer_mask])
    layer_signal = signal_corr[np.ix_(np.where(m_mask)[0][layer_mask[m_mask]],
                                       np.where(m_mask)[0][layer_mask[m_mask]])] \
                   if layer_mask.sum() > 2 else np.array([])
    
    # Simpler: compute signal corr from orientation responses within this layer
    layer_ori = ori_responses[layer_mask]
    layer_sc = np.corrcoef(layer_ori)
    layer_d = squareform(pdist(coords[layer_mask]))
    
    triu = np.triu_indices(layer_mask.sum(), k=1)
    d_vals = layer_d[triu]
    s_vals = layer_sc[triu]
    
    # Binned
    for b in range(len(distance_edges)-1):
        bm = (d_vals >= distance_edges[b]) & (d_vals < distance_edges[b+1])
        if bm.sum() > 5:
            ax.scatter(distance_centers[b], np.nanmean(s_vals[bm]), c=SUBCLASS_COLORS[sc_name], s=40)
    
    r, p = spearmanr(d_vals[~np.isnan(s_vals)], s_vals[~np.isnan(s_vals)])
    ax.set_xlabel('Distance (µm)')
    ax.set_ylabel('Signal Correlation')
    ax.set_title(f'D1: {layer} Within-Type (ρ={r:.3f}, p={p:.1e})', fontweight='bold')
    ax.axhline(0, color='gray', ls='--', alpha=0.4)

plt.tight_layout()
plt.show()

## D2: Cell-Type Composition Across Depth and Neighborhood Effects

Analyze how the inhibitory/excitatory ratio and specific subclass distribution varies with cortical depth. Test whether local neighborhood composition (e.g., VIP density) predicts functional properties of nearby excitatory cells.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# D2.1  Depth profile of cell types
# ══════════════════════════════════════════════════════════════════════

z_vals = coords[:, 2]
z_bins = np.arange(80, 360, 40)
z_centers = (z_bins[:-1] + z_bins[1:]) / 2

# Count cells per subclass per depth bin
depth_counts = {}
for sc in present_subclasses:
    sc_mask = obs['subclass_name'].values == sc
    counts, _ = np.histogram(z_vals[sc_mask], bins=z_bins)
    depth_counts[SUBCLASS_SHORT[sc]] = counts

depth_df = pd.DataFrame(depth_counts, index=z_centers)
depth_frac = depth_df.div(depth_df.sum(axis=1), axis=0)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Stacked bar of proportions
ax = axes[0]
depth_frac.plot(kind='bar', stacked=True, ax=ax,
                color=[SUBCLASS_COLORS[s] for s in present_subclasses],
                width=0.8)
ax.set_xlabel('Depth (µm)')
ax.set_ylabel('Fraction')
ax.set_title('D2: Cell-Type Proportions by Depth', fontweight='bold')
ax.legend(fontsize=7, loc='upper right')
ax.set_xticklabels([f'{int(c)}' for c in z_centers], rotation=45)

# Absolute counts
ax = axes[1]
depth_df.plot(kind='bar', ax=ax, color=[SUBCLASS_COLORS[s] for s in present_subclasses], width=0.8)
ax.set_xlabel('Depth (µm)')
ax.set_ylabel('Cell Count')
ax.set_title('D2: Cell Counts by Depth', fontweight='bold')
ax.legend(fontsize=7)
ax.set_xticklabels([f'{int(c)}' for c in z_centers], rotation=45)

# E/I ratio by depth
ax = axes[2]
exc_cols = [SUBCLASS_SHORT[s] for s in present_subclasses if 'Glut' in s]
inh_cols = [SUBCLASS_SHORT[s] for s in present_subclasses if 'Gaba' in s]
exc_counts = depth_df[exc_cols].sum(axis=1)
inh_counts = depth_df[inh_cols].sum(axis=1)
ei_ratio = exc_counts / (inh_counts + 1)  # avoid div by zero
ax.bar(range(len(z_centers)), ei_ratio, color='steelblue', width=0.8)
ax.set_xticks(range(len(z_centers)))
ax.set_xticklabels([f'{int(c)}' for c in z_centers], rotation=45)
ax.set_xlabel('Depth (µm)')
ax.set_ylabel('E/I Ratio')
ax.set_title('D2: Excitatory / Inhibitory Ratio vs Depth', fontweight='bold')

plt.tight_layout()
plt.show()

# ── 3D scatter of all cells colored by subclass ──
from mpl_toolkits.mplot3d import Axes3D
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')
for sc in present_subclasses:
    mask = obs['subclass_name'].values == sc
    ax.scatter(coords[mask, 0], coords[mask, 1], coords[mask, 2],
               alpha=0.4, s=10, c=SUBCLASS_COLORS[sc], label=SUBCLASS_SHORT[sc])
ax.set_xlabel('X (µm)')
ax.set_ylabel('Y (µm)')
ax.set_zlabel('Depth (µm)')
ax.legend(fontsize=8, loc='upper left')
ax.set_title('D2: 3D Cell Positions by Subclass', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# D2.2  Neighborhood composition → functional properties
#       For each excitatory cell, count inhibitory subtypes within radius R
#       and correlate with the excitatory cell's running modulation index (RMI)
# ══════════════════════════════════════════════════════════════════════

# Compute RMI for all cells (simplified: run vs stat overall)
run_mask_trials = var['is_running'].values.astype(bool)
mean_run_all = np.nanmean(X[:, run_mask_trials], axis=1)
mean_stat_all = np.nanmean(X[:, ~run_mask_trials], axis=1)
denom_rmi = np.abs(mean_run_all) + np.abs(mean_stat_all)
denom_rmi[denom_rmi < 1e-8] = np.nan
rmi_all = (mean_run_all - mean_stat_all) / denom_rmi

# Excitatory cells
exc_subclasses = [s for s in present_subclasses if 'Glut' in s]
exc_mask = obs['subclass_name'].isin(exc_subclasses).values
exc_indices = np.where(exc_mask)[0]
exc_coords = coords[exc_mask]

# Inhibitory subclass masks
inh_types = {'Vip': '046 Vip Gaba', 'Sst': '053 Sst Gaba', 'Pvalb': '052 Pvalb Gaba', 'Lamp5': '049 Lamp5 Gaba'}

radii = [30, 50, 100]
neighborhood_results = []

for R in radii:
    for exc_i, cell_idx in enumerate(exc_indices):
        cell_coord = coords[cell_idx]
        dists = np.sqrt(np.sum((coords - cell_coord)**2, axis=1))
        neighbors = (dists < R) & (dists > 0)
        
        record = {
            'cell_idx': cell_idx,
            'radius': R,
            'rmi': rmi_all[cell_idx],
            'osi': osi[cell_idx],
            'subclass': obs['subclass_name'].iloc[cell_idx],
            'mouse_id': obs['mouse_id'].iloc[cell_idx],
        }
        for inh_name, inh_sc in inh_types.items():
            inh_in_radius = neighbors & (obs['subclass_name'].values == inh_sc)
            record[f'n_{inh_name}'] = inh_in_radius.sum()
        
        record['n_total_neighbors'] = neighbors.sum()
        neighborhood_results.append(record)

neigh_df = pd.DataFrame(neighborhood_results)

# ── Correlations: local VIP/Sst/Pvalb density → excitatory RMI ──
fig, axes = plt.subplots(len(radii), len(inh_types), figsize=(4*len(inh_types), 4*len(radii)))

for ri, R in enumerate(radii):
    sub = neigh_df[neigh_df['radius'] == R]
    for ci, (inh_name, _) in enumerate(inh_types.items()):
        ax = axes[ri, ci]
        col = f'n_{inh_name}'
        valid = ~np.isnan(sub['rmi'])
        x = sub.loc[valid, col].values
        y = sub.loc[valid, 'rmi'].values
        
        ax.scatter(x, y, alpha=0.15, s=10, c='gray')
        
        # Binned mean
        unique_counts = sorted(np.unique(x))
        for uc in unique_counts:
            mask_c = x == uc
            if mask_c.sum() >= 5:
                ax.scatter(uc, np.mean(y[mask_c]), c='red', s=60, zorder=5, edgecolors='k')
        
        if np.std(x) > 0:
            r_corr, p_corr = spearmanr(x, y)
            ax.set_title(f'{inh_name} (R={R}µm)\nρ={r_corr:.3f}, p={p_corr:.2e}',
                        fontweight='bold', fontsize=10)
        ax.set_xlabel(f'# {inh_name} within {R}µm')
        if ci == 0:
            ax.set_ylabel('Excitatory cell RMI')

plt.suptitle('D2: Local Inhibitory Density → Excitatory Running Modulation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Summary statistics ──
print("=== Neighborhood composition effects (R=50µm) ===")
sub50 = neigh_df[neigh_df['radius'] == 50]
for inh_name in inh_types:
    col = f'n_{inh_name}'
    valid = ~np.isnan(sub50['rmi'])
    r_corr, p_corr = spearmanr(sub50.loc[valid, col], sub50.loc[valid, 'rmi'])
    print(f"  {inh_name} count vs RMI: ρ={r_corr:.3f}, p={p_corr:.2e}")